# Question 2: News Topic Clustering and Semantic Embedding Comparison

**Course practical:** NLP / Machine Learning exam  
**Task:** Automatically organize news articles into meaningful topics without predefined labels.  
**Dataset:** Kaggle News Category Dataset by Rishabh Misra.

.

## Rubric Map

| Exam part | What this notebook shows |
| --- | --- |
| (a) Data preparation | Loads the dataset, displays first 10 records, explains columns, computes document statistics, identifies noise. |
| (b) Preprocessing | Cleans text with regex, lowercases, tokenizes, removes stopwords, normalizes word forms, shows before/after examples. |
| (c) NLP exploration | Displays top words, bigrams, and trigrams with charts and explanation. |
| (d) Representation | Builds Bag-of-Words and TF-IDF matrices, compares weighting and sparsity. |
| (e) Similarity | Uses cosine similarity on TF-IDF vectors and displays most similar article pairs. |
| (f) Clustering | Applies K-Means, uses elbow/silhouette selection, displays cluster samples and topic labels. |
| (g) Transformers | Generates SentenceTransformer embeddings and clusters them with K-Means. |
| (h) Evaluation | Compares TF-IDF and transformer clustering with keywords, labels, examples, and silhouette scores. |

## 1. Setup

The companion script `q2_news_clustering.py` generated the final outputs used in this notebook. The notebook also includes runnable code so the logic can be inspected and repeated.

In [2]:
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Image, display
from nltk import ngrams
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'News_Category_Dataset_v3.json').exists() and (BASE_DIR / 'News_Category').exists():
    BASE_DIR = BASE_DIR / 'News_Category'

DATA_FILE = BASE_DIR / 'News_Category_Dataset_v3.json'
OUTPUT_DIR = BASE_DIR / 'outputs'
RANDOM_STATE = 42

pd.set_option('display.max_colwidth', 140)
plt.style.use('seaborn-v0_8-whitegrid')
print('Dataset:', DATA_FILE)
print('Outputs:', OUTPUT_DIR)

Dataset: c:\Users\USER\Desktop\ACADEMICS\nlp\EXAM\News_Category\News_Category_Dataset_v3.json
Outputs: c:\Users\USER\Desktop\ACADEMICS\nlp\EXAM\News_Category\outputs


## (a) Data Preparation and Understanding

The dataset contains news article metadata and text. For topic analysis, the most important columns are:

- `headline`: short, high-signal summary of the article topic.
- `short_description`: adds more context to the headline.
- `category`: original editorial category. It is not used as a label during clustering, but it helps validate whether clusters make sense.
- `date`, `authors`, and `link`: useful metadata, but less central to text clustering.

In [3]:
def repair_mojibake(value):
    """Repair common UTF-8 text accidentally decoded as Latin-1."""
    if not isinstance(value, str) or '?' not in value:
        return value
    try:
        return value.encode('latin-1').decode('utf-8')
    except UnicodeError:
        return value

raw_df = pd.read_json(DATA_FILE, lines=True)
raw_df['headline'] = raw_df['headline'].fillna('').apply(repair_mojibake)
raw_df['short_description'] = raw_df['short_description'].fillna('').apply(repair_mojibake)
raw_df['document'] = (raw_df['headline'] + ' ' + raw_df['short_description']).str.strip()
raw_df = raw_df[raw_df['document'].str.len() > 0].copy()

raw_df[['headline', 'category', 'short_description', 'authors', 'date']].head(10)

,headline,category,short_description,authors,date
0,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters,U.S. NEWS,Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ord...,"Carla K. Johnson, AP",2022-09-23
1,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video",U.S. NEWS,"He was subdued by passengers and crew when he fled to the back of the aircraft after the confrontation, according to the U.S. attorney's...",Mary Papenfuss,2022-09-23
2,23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23),COMEDY,"""Until you have a dog you don't understand what could be eaten.""",Elyse Wanshel,2022-09-23
3,The Funniest Tweets From Parents This Week (Sept. 17-23),PARENTING,"""Accidentally put grown-up toothpaste on my toddler’s toothbrush and he screamed like I was cleaning his teeth with a Carolina Reaper di...",Caroline Bologna,2022-09-23
4,Woman Who Called Cops On Black Bird-Watcher Loses Lawsuit Against Ex-Employer,U.S. NEWS,Amy Cooper accused investment firm Franklin Templeton of unfairly firing her and branding her a racist after video of the Central Park e...,Nina Golgowski,2022-09-22
5,Cleaner Was Dead In Belk Bathroom For 4 Days Before Body Found: Police,U.S. NEWS,The 63-year-old woman was seen working at the South Carolina store on Thursday. She was found dead Monday after her family reported her ...,,2022-09-22
6,Reporter Gets Adorable Surprise From Her Boyfriend While Live On TV,U.S. NEWS,"""Who's that behind you?"" an anchor for New York’s PIX11 asked journalist Michelle Ross as she finished up an interview.",Elyse Wanshel,2022-09-22
7,Puerto Ricans Desperate For Water After Hurricane Fiona’s Rampage,WORLD NEWS,More than half a million people remained without water service three days after the storm lashed the U.S. territory.,"DÁNICA COTO, AP",2022-09-22
8,How A New Documentary Captures The Complexity Of Being A Child Of Immigrants,CULTURE & ARTS,"In ""Mija,"" director Isabel Castro combined music documentaries with the style of ""Euphoria"" and ""Clueless"" to tell a more nuanced immigr...",Marina Fang,2022-09-22
9,Biden At UN To Call Russian War An Affront To Body's Charter,WORLD NEWS,White House officials say the crux of the president's visit to the U.N. this year will be a full-throated condemnation of Russia and its...,"Aamer Madhani, AP",2022-09-21


In [4]:
# Use the final sampled output from the script so notebook results match the submitted report.
clustered_df = pd.read_csv(OUTPUT_DIR / 'q2_clustered_news_sample.csv')
tfidf_summary = pd.read_csv(OUTPUT_DIR / 'q2_tfidf_cluster_summary.csv')
embedding_summary = pd.read_csv(OUTPUT_DIR / 'q2_embedding_cluster_summary.csv')
similarity_pairs = pd.read_csv(OUTPUT_DIR / 'q2_similarity_pairs.csv')
elbow_scores = pd.read_csv(OUTPUT_DIR / 'q2_elbow_scores.csv')
model_comparison = pd.read_csv(OUTPUT_DIR / 'q2_model_comparison.csv')

clean_tokens = clustered_df['clean_text'].fillna('').astype(str).str.split().tolist()
statistics = pd.DataFrame({
    'Statistic': ['Full dataset documents', 'Documents used in final run', 'Average cleaned document length', 'Vocabulary size after tokenization'],
    'Value': [
        len(raw_df),
        len(clustered_df),
        round(np.mean([len(tokens) for tokens in clean_tokens]), 2),
        len(set(token for tokens in clean_tokens for token in tokens))
    ]
})
statistics

,Statistic,Value
0,Full dataset documents,209522.00
1,Documents used in final run,8000.00
2,Average cleaned document length,15.39
3,Vocabulary size after tokenization,16618.00


### Noise and Inconsistencies

The raw news text contains several issues that can reduce clustering quality:

1. **Punctuation and symbols:** headlines contain parentheses, apostrophes, commas, and other symbols that create extra token variants.
2. **Capitalization differences:** words like `Trump`, `TRUMP`, and `trump` should be treated as the same token.
3. **Short or empty descriptions:** articles with little text produce weak vectors and may be pulled into broad clusters.
4. **Named entities and source style language:** names, dates, and repeated media wording can dominate clusters if not normalized.
5. **Encoding artifacts:** characters such as `???` sometimes appear when text encoding is misread; these should be repaired.

The category chart below is not used for training; it is shown only to understand the sampled data composition.

![Top original news categories](outputs/q2_category_distribution.png)

## (b) Text Preprocessing and Normalization

The cleaning stage prepares text for vectorization and clustering. It lowercases text, removes URLs/punctuation/special characters, tokenizes words, removes stopwords, and applies a small lemmatization fallback.

In [5]:
EXTRA_STOPWORDS = {
    'said', 'say', 'says', 'new', 'year', 'years', 'day', 'week', 'time',
    'people', 'just', 'like', 'huffpost'
}
STOPWORDS = set(ENGLISH_STOP_WORDS).union(EXTRA_STOPWORDS)


def simple_lemma(token):
    if len(token) > 4 and token.endswith('ies'):
        return token[:-3] + 'y'
    if len(token) > 5 and token.endswith('ing'):
        return token[:-3]
    if len(token) > 4 and token.endswith('ed'):
        return token[:-2]
    if len(token) > 3 and token.endswith('s'):
        return token[:-1]
    return token


def clean_and_tokenize(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = re.findall(r'[a-z]+', text)
    return [simple_lemma(token) for token in tokens if token not in STOPWORDS and len(token) > 2]

preprocess_examples = raw_df[['document']].head(4).copy()
preprocess_examples['tokens'] = preprocess_examples['document'].apply(clean_and_tokenize)
preprocess_examples['cleaned_text'] = preprocess_examples['tokens'].apply(lambda row: ' '.join(row))
preprocess_examples.rename(columns={'document': 'original_text'})

,original_text,tokens,cleaned_text
0,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters Health experts said it is too early to predict whether dema...,"[million, american, roll, sleeve, omicron, target, covid, booster, health, expert, early, predict, demand, match, million, dose, booster...",million american roll sleeve omicron target covid booster health expert early predict demand match million dose booster order fall
1,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video He was subdued by passengers and crew when he ...","[american, airline, flyer, charg, bann, life, punch, flight, attendant, video, subdu, passenger, crew, fled, aircraft, confrontation, ac...",american airline flyer charg bann life punch flight attendant video subdu passenger crew fled aircraft confrontation accord attorney off...
2,"23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23) ""Until you have a dog you don't understand what could be eaten.""","[funniest, tweet, cat, dog, sept, dog, don, understand, eaten]",funniest tweet cat dog sept dog don understand eaten
3,"The Funniest Tweets From Parents This Week (Sept. 17-23) ""Accidentally put grown-up toothpaste on my toddler’s toothbrush and he screame...","[funniest, tweet, parent, sept, accidentally, grown, toothpaste, toddler, toothbrush, scream, clean, teeth, carolina, reaper, dipp, taba...",funniest tweet parent sept accidentally grown toothpaste toddler toothbrush scream clean teeth carolina reaper dipp tabasco sauce


**Why preprocessing helps clustering:** after cleaning, articles about the same topic share more comparable vocabulary. For example, punctuation and capitalization no longer split the same word into different forms, and stopword removal leaves more topic-bearing terms.

## (c) NLP Exploration and Pattern Discovery

Frequent terms give a quick view of dominant topics. N-grams are even more useful because many news topics appear as phrases, such as `donald trump`, `white house`, `health care`, and `climate change`.

In [6]:
word_counts = Counter(token for tokens in clean_tokens for token in tokens).most_common(10)
bigram_counts = Counter(' '.join(tokens[i:i+2]) for tokens in clean_tokens for i in range(len(tokens)-1)).most_common(10)
trigram_counts = Counter(' '.join(tokens[i:i+3]) for tokens in clean_tokens for i in range(len(tokens)-2)).most_common(10)

display(pd.DataFrame(word_counts, columns=['term', 'count']))
display(pd.DataFrame(bigram_counts, columns=['bigram', 'count']))
display(pd.DataFrame(trigram_counts, columns=['trigram', 'count']))

,term,count
0,trump,788
1,photo,589
2,make,450
3,want,439
4,way,428
5,look,366
6,life,362
7,thing,346
8,know,346
9,need,340


,bigram,count
0,donald trump,266
1,white house,74
2,hillary clinton,69
3,health care,51
4,climate change,51
5,supreme court,46
6,unit state,39
7,social media,37
8,york city,35
9,super bowl,33


,trigram,count
0,want sure check,24
1,twitter facebook tumblr,20
2,facebook tumblr pinterest,20
3,sure check style,17
4,check style twitter,17
5,style twitter facebook,17
6,photo want sure,12
7,tumblr pinterest instagram,12
8,twitter facebook pinterest,10
9,black live matter,10


### Visual Exploration

![Top words](outputs/q2_top_words.png)

![Top bigrams](outputs/q2_top_bigrams.png)

![Top trigrams](outputs/q2_top_trigrams.png)

**Interpretation:** The most common words and phrases point to repeated themes in the sample. Political terms such as `donald trump`, public-policy phrases such as `health care`, and social topics such as `black live matter` help explain why clusters later form around politics, health/social issues, lifestyle, entertainment, and family-related content.

## (d) Text Representation

Two traditional representations are created:

- **Bag-of-Words:** stores raw term counts.
- **TF-IDF:** stores weighted scores that reduce the importance of very common terms and increase the importance of distinctive terms.

TF-IDF is usually better for similarity and clustering because a rare, topic-specific term should matter more than a general repeated word.

In [7]:
bow_vectorizer = CountVectorizer(max_features=5000, min_df=3, max_df=0.85)
bow_matrix = bow_vectorizer.fit_transform(clustered_df['clean_text'].fillna(''))

tfidf_vectorizer = TfidfVectorizer(max_features=5000, min_df=3, max_df=0.85)
tfidf_matrix = tfidf_vectorizer.fit_transform(clustered_df['clean_text'].fillna(''))

representation_summary = pd.DataFrame({
    'Representation': ['Bag-of-Words', 'TF-IDF'],
    'Documents': [bow_matrix.shape[0], tfidf_matrix.shape[0]],
    'Features': [bow_matrix.shape[1], tfidf_matrix.shape[1]],
    'Non-zero values': [bow_matrix.nnz, tfidf_matrix.nnz],
    'Sparsity (%)': [
        round(100 * (1 - bow_matrix.nnz / (bow_matrix.shape[0] * bow_matrix.shape[1])), 2),
        round(100 * (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])), 2)
    ]
})
representation_summary

,Representation,Documents,Features,Non-zero values,Sparsity (%)
0,Bag-of-Words,8000,5000,95609,99.76
1,TF-IDF,8000,5000,95609,99.76


In [8]:
# Example TF-IDF weights for one article.
doc_index = 0
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
row = tfidf_matrix[doc_index]
nonzero = row.nonzero()[1]
weights = row.data
example_weights = pd.DataFrame({'term': feature_names[nonzero], 'tfidf_score': weights})
example_weights = example_weights.sort_values('tfidf_score', ascending=False).head(12)
print(clustered_df.loc[doc_index, 'headline'])
example_weights

Rick Snyder Says He's Releasing All His Emails About The Flint Water Crisis


,term,tfidf_score
3,flint,0.389412
0,rick,0.384582
7,document,0.351815
1,releas,0.325930
5,crisi,0.323114
6,clear,0.319164
2,email,0.312081
4,water,0.302113
8,public,0.274801


## (e) Document Similarity Analysis

Cosine similarity measures whether two TF-IDF vectors point in a similar direction. It is useful for finding articles that discuss related ideas, even if their total lengths differ.

In [9]:
def shared_terms_for_pair(row):
    a_terms = set(clustered_df.loc[int(row['doc_a']), 'clean_text'].split())
    b_terms = set(clustered_df.loc[int(row['doc_b']), 'clean_text'].split())
    return ', '.join(sorted(a_terms.intersection(b_terms))[:12])

similarity_display = similarity_pairs.copy()
similarity_display['shared_terms'] = similarity_display.apply(shared_terms_for_pair, axis=1)
similarity_display[['doc_a', 'doc_b', 'score', 'headline_a', 'headline_b', 'category_a', 'category_b', 'shared_terms']]

,doc_a,doc_b,score,headline_a,headline_b,category_a,category_b,shared_terms
0,21,120,0.423033,Global Forgiveness Day: 5 Celebs Who Forgave Their Exes,Celebrity Exes: 6 Celeb Exes We'd Like To Cast In A Reality Show,DIVORCE,DIVORCE,"celeb, exe"
1,252,137,0.410043,Republicans Have Always Been At War With The New York Times,The New York Times Just Provided A Massive Platform For Transphobia,POLITICS,QUEER VOICES,"time, york"
2,192,280,0.391524,Amber Rose's 19 Sexiest Social Media Snaps,Amber Rose Delivers Impassioned Speech About The First Time She Was Slut-Shamed,ENTERTAINMENT,WOMEN,"amber, rose"
3,230,236,0.357609,"Laura Nekrasoe, Waitress, Swears By DIY Face Masks For Dry Skin",7 DIY Revitalizing Beauty Tips for Hair and Skin,STYLE & BEAUTY,STYLE & BEAUTY,"diy, skin"
4,245,110,0.328358,"Dad, Why Is that Man Wearing a Red Flower?",Hatchimals Are So Hard To Find That Parents Are Making Santa Explain,PARENTS,PARENTS,"explain, man, red, turn"
5,204,220,0.328232,"Russian Trolls Didn't Need To Infiltrate The American Media, Because We Let Them In",Bury Lenin's Body: The Symbol of Communism Should No Longer Mock Humanity,MEDIA,WORLDPOST,russian
6,84,59,0.310038,Death Toll From Italy Bridge Collapse Climbs To 37,America's Most Dangerous Bridges (PHOTOS),WORLD NEWS,TRAVEL,bridge
7,112,74,0.297048,Elin Nordegren: 10 Things She's Done Since The Tiger Woods Scandal,"Joe Nocera: Another Week, Another Banking Scandal",DIVORCE,BUSINESS,scandal
8,40,258,0.282124,"Georgia DA Could Bury Trump With His Own Words, Says Former Fed Prosecutor","Nirvana Jennette, Mom Forced Out Of Church For Breastfeeding, Aims To Change Georgia Law [UPDATED]",POLITICS,PARENTING,georgia
9,288,144,0.278798,Marcus Garvey's Message And Why A Pardon For Him Matters,Buckminster Fuller's Vision of a World That Works for Everyone,BLACK VOICES,ENTERTAINMENT,vision


![Top cosine similarity scores](outputs/q2_similarity_scores.png)

**Interpretation:** The highest-ranked pairs tend to share important names, topic phrases, or category-specific vocabulary. This demonstrates why TF-IDF cosine similarity can identify related articles without using labels.

## (f) Machine Learning: K-Means Clustering

K-Means is trained on the TF-IDF matrix. The number of clusters is selected using inertia and cosine silhouette scores. In this final run, the best silhouette score among the tested values occurs at **k = 9**.

In [10]:
elbow_scores

,k,inertia,silhouette
0,2,7935.959858,0.002360
1,3,7908.009430,0.003762
2,4,7894.054350,0.004348
3,5,7882.962910,0.005167
4,6,7872.354116,0.005367
5,7,7862.986461,0.006144
6,8,7850.777745,0.006533
7,9,7843.143338,0.006892
8,10,7835.143378,0.004952


![TF-IDF elbow and silhouette plot](outputs/q2_elbow_tfidf.png)

In [11]:
tfidf_summary

,cluster,size,topic_label,top_keywords,sample_headlines
0,0,5050,"Style, photos, and lifestyle media","make, look, know, life, want, women, thing, video, need, state",Rick Snyder Says He's Releasing All His Emails About The Flint Water Crisis | Mark Sanchez Pick Six: Bryan Scott Returns Interception Fo...
1,1,521,"Style, photos, and lifestyle media","photo, check, style, home, pinterest, twitter, facebook, look, fashion, wedd",Check Out Jiff The Pomeranian's New Move | 10 Great State Fairs For Deep-Fried Fun (PHOTOS) | Russian Cargo Ship Docks At International ...
2,2,420,Health and social issues,"american, health, black, care, women, live, mental, man, study, country",Watch GOP Lawmakers Run Away When Asked If They Actually Read The Health Care Bill | How Russia Often Benefits When Julian Assange Revea...
3,3,100,Politics and elections,"clinton, hillary, trump, sander, campaign, bernie, donald, email, democratic, election",Hillary Clinton's Paid Speeches Were Totally At Odds With Her 2016 Platform | Donald Trump Is 'Starting To Agree' Hillary Clinton Should...
4,4,368,Video and entertainment,"world, watch, cup, thing, live, need, make, art, video, look",The Happiest Countries In The World (INFOGRAPHIC) | Mediterranean Horrors: Fortress Europe's Vast Moat | World's Dumbest Hijack Attempt?
5,5,237,"Education, schools, and children","school, really, high, want, children, student, teacher, kid, think, college","We Tried It: SLT Yoga | Justin Theroux Really, Really Hates Teva Sandals | GOP Senate Candidate David Perdue Exaggerates His Father's Ro..."
6,6,271,Parenting and family,"kid, parent, mom, children, child, dad, need, want, teach, baby","Mom Of Transgender Teen Describes Her Experience As A Gift | 10 Reasons Why It's OK To Spend New Year's Eve At Home, As Told In GIFs | G..."
7,7,552,Politics and elections,"trump, donald, president, republican, gop, white, house, campaign, democrat, russia",Insensitive Washington Times Columnist Puts His Idiocy On Display | The Problem With Paternalizing Disabled People To Protest Donald Tru...
8,8,481,Personal life and relationships,"way, best, make, life, thing, work, love, better, don, want",April Superfoods: 5 In-Season Picks | A Heart-to-Heart With the LGBTQ Community | Gay One-Night Stands: Are They Keeping You From A Real...


![TF-IDF cluster sizes](outputs/q2_cluster_sizes_tfidf.png)

**Cluster interpretation:** The top keywords and sample headlines give each cluster a meaningful topic label. TF-IDF clusters are highly interpretable because the most important centroid terms can be inspected directly.

## (g) Transformer-Based Embeddings and Comparison

The transformer stage uses `SentenceTransformer all-MiniLM-L6-v2` to encode each article into a dense semantic vector. Unlike TF-IDF, transformer embeddings can place documents close together when they have similar meaning even if they do not share many exact words.

In [12]:
def transformer_or_svd_embeddings(clean_text, tfidf_matrix):
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('all-MiniLM-L6-v2')
        embeddings = model.encode(clean_text, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
        return np.asarray(embeddings), 'SentenceTransformer all-MiniLM-L6-v2'
    except Exception as exc:
        svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
        embeddings = svd.fit_transform(tfidf_matrix)
        return embeddings, f'TruncatedSVD fallback: {type(exc).__name__}: {exc}'

# The final script run used real SentenceTransformer embeddings.
model_comparison

,representation,k,silhouette
0,TF-IDF,9,0.006892
1,SentenceTransformer all-MiniLM-L6-v2,9,0.044826


In [13]:
embedding_summary

,cluster,size,topic_label,common_terms,sample_headlines
0,0,766,World news and international affairs,"kill, police, state, gun, law, death, arrest, shoot",Rick Snyder Says He's Releasing All His Emails About The Flint Water Crisis | Dustin Hoffman Accusers Speak Out About Alleged Abuse In J...
1,1,842,Health and social issues,"life, way, make, need, health, help, feel, thing",Non-Reactive Listening | Who Are You Letting Into Your Head? | We Tried It: SLT Yoga
2,2,1247,Video and entertainment,"video, star, watch, love, game, live, music, film",Mark Sanchez Pick Six: Bryan Scott Returns Interception For Touchdown In Jets-Bills (GIF) | Check Out Jiff The Pomeranian's New Move | G...
3,3,1226,Politics and elections,"trump, donald, president, republican, gop, obama, house, clinton",Insensitive Washington Times Columnist Puts His Idiocy On Display | The Problem With Paternalizing Disabled People To Protest Donald Tru...
4,4,655,Parenting and family,"kid, parent, children, mom, child, baby, dad, thing","Mom Of Transgender Teen Describes Her Experience As A Gift | Pregnant Nurse, Dreonna Breton, Fired For Refusing Flu Vaccine | 10 Reasons..."
5,5,621,"Style, photos, and lifestyle media","food, recipe, eat, best, make, way, photo, restaurant",April Superfoods: 5 In-Season Picks | 10 Great State Fairs For Deep-Fried Fun (PHOTOS) | How To Pit Cherries Without Using A Cherry Pitter
6,6,618,Health and social issues,"women, gay, marriage, woman, right, men, divorce, american",Barbies May Limit Girls' Career Aspirations (STUDY) | Will Smith Joins Wife Jada Pinkett Smith In Oscars Boycott | Queen Elizabeth Gay R...
7,7,1188,World news and international affairs,"world, state, need, travel, want, city, change, make","I Visited 29 States In 90 Days For Just $3,600 | 'Cash Mobs' Use Social Media To Splurge In Locally Owned Stores | Why Los Angeles Sends..."
8,8,837,"Style, photos, and lifestyle media","photo, look, fashion, style, home, art, best, wedd","Selena Gomez's AMAs Dress Is Very Sexy | Ashton Kutcher And Mila Kunis' Matching Olympics Outfits Are So Damn Cute, They Deserve A Medal..."


![Transformer cluster sizes](outputs/q2_cluster_sizes_embedding.png)

**Comparison examples:** The transformer clusters are more balanced and semantically coherent. For example, articles about politics can cluster together even when one headline uses `campaign`, another uses `president`, and another uses `GOP`. TF-IDF may miss this if exact vocabulary overlap is weak.

## (h) Evaluation and Insight

Since clustering is unsupervised, evaluation is qualitative and structural rather than simple accuracy. We inspect:

- top keywords per cluster,
- sample headlines per cluster,
- meaningful topic labels,
- cluster sizes,
- cosine silhouette scores,
- and comparison between TF-IDF and transformer embeddings.

In [14]:
model_comparison

,representation,k,silhouette
0,TF-IDF,9,0.006892
1,SentenceTransformer all-MiniLM-L6-v2,9,0.044826


![Model comparison](outputs/q2_model_comparison.png)

### Conclusion

The transformer embedding approach produced the stronger clustering result in this run. The TF-IDF silhouette score was about **0.0069**, while the SentenceTransformer embedding silhouette score was about **0.0448**. Both values are low because short news snippets are broad and overlapping, but the transformer score is clearly higher.

**Strength of TF-IDF:** easy to interpret using top words from cluster centroids.  
**Weakness of TF-IDF:** depends heavily on exact word overlap.  
**Strength of transformer embeddings:** captures semantic similarity and context.  
**Weakness of transformer embeddings:** slower and more computationally expensive.

Overall, TF-IDF is useful for explanation, while transformer embeddings provide better semantic grouping.

## Companion Files

The following files support this notebook:

- `q2_news_clustering.py`: complete reproducible script.
- `outputs/q2_report.md`: written report answer.
- `outputs/q2_tfidf_cluster_summary.csv`: TF-IDF cluster labels, keywords, and examples.
- `outputs/q2_embedding_cluster_summary.csv`: transformer cluster labels, terms, and examples.
- `outputs/q2_similarity_pairs.csv`: most similar TF-IDF document pairs.
- `outputs/q2_elbow_tfidf.png`: elbow/silhouette visual.
- `outputs/q2_model_comparison.csv`: TF-IDF vs transformer silhouette comparison.